In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
#RR
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as random
import optax
from tqdm import tqdm
from utils import DataLoader
from flax import nnx
from flax.nnx.training.metrics import Metric, Average

In [ ]:
class MLP(nnx.Module): #da rimettere in utils
    """
    MLP class
    """
    input_dim: int
    output_dim: int = 1
    hidden_dim: int = 8
    num_hidden_layers: int = 1

    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int, num_hidden_layers: int, *, rngs: nnx.Rngs = nnx.Rngs(0)):
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.num_hidden_layers = num_hidden_layers
        if num_hidden_layers > 0:
            self.hidden_layers = [nnx.Linear(self.input_dim, self.hidden_dim, rngs=rngs)] + \
                [nnx.Linear(self.hidden_dim, self.hidden_dim, rngs=rngs) for _ in range(self.num_hidden_layers - 1)]
            self.linear = nnx.Linear(hidden_dim, output_dim, rngs=rngs)
        else:
            self.linear = nnx.Linear(self.input_dim, self.output_dim, rngs=rngs)

    def __call__(self, x: jax.Array):
        if self.num_hidden_layers > 0:
            for i in range(self.num_hidden_layers):
                x = self.hidden_layers[i](x)
                x = nnx.relu(x) 
            x = self.linear(x) 
        else:
            x = self.linear(x)
        return x

In [77]:
#GENERAZIONE DEL DATASET
key = random.key(0)
f_to_learn = lambda mu, l, k, x: (mu+l+k)*x
#f_to_learn = lambda mu, k, l, x: jnp.sin(2*mu*jnp.pi*x) + k + jnp.exp(-l*x)

N = 1000
key, subkey = random.split(key) 
x = random.uniform(subkey, (N,), minval=-10, maxval=10)
key, subkey = random.split(key) 
mu = random.uniform(subkey, (N,), minval=-2, maxval=2)
key, subkey = random.split(key) 
l = random.uniform(subkey, (N,), minval=0, maxval=1)
key, subkey = random.split(key) 
k = random.uniform(subkey, (N,), minval=-2, maxval=2)
y = f_to_learn(mu, l, k, x)
X = jnp.stack([mu, l, k, x], axis=1)

# DIVISIONE DEL DATASET IN TRAIN E TEST E CREAZIONE DEI DATALOADER
split_idx = int(N * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
train_dataloader = DataLoader(X_train, y_train, batch_size=8, shuffle=True)
test_dataloader = DataLoader(X_test, y_test, batch_size=8, shuffle=False)

In [78]:
@nnx.jit
@nnx.vmap #(equivalente a nnx.vmap(assign_params, in_axes=0, out_axes=0))
def assign_parameters(state, parameters):
    """
    Assigns the parameters from an array to the state. state and parameters must be batched with the same first dimension.
    """
    parameters = parameters.squeeze()
    flat_state = nnx.to_flat_state(state)

    # TODO: Check if the number of parameters matches
    
    i=0
    for (key, param) in flat_state:
        param.value = parameters[i: i + param.value.size].reshape(param.value.shape)
        i += param.value.size   
    return state

#qui niente jit
def apply(network, parameters, x):
    """
    Applies the network with the given parameters to the input x.
    The parameters must be batched with the same first dimension as x.
    """

    graphdef, state = nnx.split(network)
    flat_state = nnx.to_flat_state(state)

    for (key, param) in flat_state:
        param.value = jnp.repeat(param.value[None, ...], parameters.shape[0], axis=0)

    state = assign_parameters(state, parameters)

    @nnx.vmap
    def apply_state(state, x):
        modified_network = nnx.merge(graphdef, state)
        return modified_network(x)
    
    y = apply_state(state, x)

    return y

@nnx.jit
def train_step(hypernetwork, targetnetwork, hyperparams, x, y, optimizer):
    """
    Performs a single training step."""
    def loss_fn(hypernetwork, hyperparams, x, y):
        w = hypernetwork(hyperparams)
        pred = apply(targetnetwork, w, x)
        loss = jnp.mean(optax.l2_loss(pred, y))
        return loss
    loss, grads = nnx.value_and_grad(loss_fn)(hypernetwork, hyperparams, x, y)
    optimizer.update(grads)
    return loss

@nnx.jit
def evaluation_step(hypernetwork, targetnetwork, hyperparams, x, y):
    """
    Performs a single evaluation step.
    """
    def loss_fn(hypernetwork, hyperparams, x, y):
        w = hypernetwork(hyperparams)
        pred = apply(targetnetwork, w, x)
        loss = jnp.mean(optax.l2_loss(pred, y))
        return loss
    loss = loss_fn(hypernetwork, hyperparams, x, y)
    return loss

In [ ]:
hypernetwork = MLP(input_dim=3, output_dim=25, hidden_dim=8, num_hidden_layers=2, rngs=nnx.Rngs(0))
targetnetwork = MLP(input_dim=1, output_dim=1, hidden_dim=8, num_hidden_layers=1, rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(hypernetwork, optax.adam(learning_rate=1e-3))
training_loss = Average(argname='loss')
test_loss = Average(argname='loss')
epochs = 200
pbar = tqdm(range(epochs))
for epoch in pbar:
    pbar.set_description(f"Epoch {epoch+1}")
    training_loss.reset()
    test_loss.reset()
    for data, label in train_dataloader:
        hyperparams = data[:, :-1] # mu, l, k
        x = data[:, -1:] # x
        loss_step = train_step(hypernetwork, targetnetwork, hyperparams, x, label, optimizer)
        training_loss.update(loss = loss_step.item()) #TODO: CONTROLLARE SE FUNZIONA (IN PARTICOLARE SE TIENE CONTO CHE L'ULTIMO BATCH POTREEBBE ESSERE PIU' PICCOLO)
    for data, label in test_dataloader:
        hyperparams = data[:, :-1]
        x = data[:, -1:]
        loss_step = evaluation_step(hypernetwork, targetnetwork, hyperparams, x, label)
        test_loss.update(loss = loss_step.item())
    pbar.set_postfix({"training loss": training_loss.compute(), "test loss": test_loss.compute()})

Epoch 1:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 56:  28%|██▊       | 55/200 [00:26<01:06,  2.19it/s, training loss=0.060622323, test loss=0.11890806]

In [42]:
apply = nnx.jit(apply)
w = jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])
x = jnp.array([[1.0], [2.0], [3.0], [4.0]])
apply(targetnetwork, w, x)

# COSI FUNZIONA, SE USIAMO QUESTA VERSIONE JITTATA DENTRO A TRAIN_STEP SI ROMPE

Array([[ 3.],
       [11.],
       [23.],
       [39.]], dtype=float32)